In [1]:
import duckdb 
import pandas as pd

# Exploration of admissions data

In [ ]:
# Create a duckdb session/connection
admissions_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/admissions.sql", "r") as f:
    sql: str = f.read()

admissions_result: pd.DataFrame = admissions_con.execute(sql).df() # remember to fetch result using df()
admissions_result

,avg(subject_count)
0,2.443603


# Exploration of diagnoses data

In [ ]:
diagnoses_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/diagnoses_sql/active_cancer.sql", "r") as f:
    sql: str = f.read()

diagnoses_result: pd.DataFrame = diagnoses_con.execute(sql).df() # remember to fetch result using df()
diagnoses_result

,count(DISTINCT subject_id)
0,29369


In [ ]:
diagnoses_history_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/diagnoses_sql/personal_history_cancer.sql", "r") as f:
    sql: str = f.read()

diagnoses_history_result: pd.DataFrame = diagnoses_history_con.execute(sql).df() # remember to fetch result using df()
diagnoses_history_result

,subject_id
0,11378357
1,11384260
2,11384293
3,11402448
4,11407323
...,...
32394,19557768
32395,19574332
32396,19595058
32397,19631158


In [2]:
total_diagnoses_patients_con: duckdb.DuckDBPyConnection = duckdb.connect()

diagnoses_sql_queries = ["active_cancer", "personal_history_cancer"]

# Perform SQL queries so multiple files can be loaded. 
for query in diagnoses_sql_queries:
    with open(f"exploration_sql_files/diagnoses_sql/{query}.sql", "r") as f:
        sql: str = f.read()
        total_diagnoses_patients_con.execute(sql).df()
        
with open(f"exploration_sql_files/diagnoses_sql/history_and_active.sql", "r") as f:
    total_cancer_patients_sql: str = f.read()


total_diagnoses_result: pd.DataFrame = total_diagnoses_patients_con.execute(total_cancer_patients_sql).df()
total_diagnoses_result

,subject_id
0,12240787
1,12243782
2,12252440
3,12254879
4,12261942
...,...
42494,19418137
42495,19517365
42496,19602017
42497,19627197


# Exploration of prescriptions data

1. Look at how many patients have had antineoplasts or specific drugs across the entire dataset. 
2. Perform 1. but within the total number of cancer patients. 

In [ ]:
prescriptions_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/prescriptions_sql/prescriptions_count.sql", "r") as f:
    sql: str = f.read()

prescriptions_result: pd.DataFrame = prescriptions_con.execute(sql).df() # remember to fetch result using df()
prescriptions_result

,subject_id
0,10027393
1,10035631
2,10047727
3,10102610
4,10109081
...,...
3217,19880967
3218,19900111
3219,19930554
3220,19988166


# Find intersection of diagnoses and prescriptions 

In [14]:
common_diag_prescript_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(prescriptions_result["subject_id"])]
common_diag_prescript_patients

,subject_id
1,10979480
4,11036723
13,11057908
28,11124859
31,11173335
...,...
45982,13568398
46048,13099797
46476,11890400
46787,11296156


# Find cancer-related procedures

In [ ]:
procedures_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/procedures_sql/procedures_count.sql", "r") as f:
    sql: str = f.read()

procedures_result: pd.DataFrame = procedures_con.execute(sql).df() # remember to fetch result using df()
procedures_result

,subject_id
0,10002155
1,10003019
2,10035168
3,10064049
4,10098008
...,...
2886,19851462
2887,19860377
2888,19906882
2889,19915727


# Find intersection of cancer diagnoses and procedures

In [24]:
common_diag_procedure_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(procedures_result["subject_id"])]
common_diag_procedure_patients

,subject_id
1,10979480
4,11036723
23,11204099
28,11124859
36,11234187
...,...
45853,15335155
45879,12204179
45986,13736767
46040,18529025


# Get study counts for echocardiography

Do note that there are different populations in `echo-study-list.csv` and `structured-measurement.csv` so we use the latter. 

In [ ]:
echo_study_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/echo_sql/study_counts.sql", "r") as f:
    sql: str = f.read()

echo_study_result: pd.DataFrame = echo_study_con.execute(sql).df() # remember to fetch result using df()
echo_study_result

,n_patients,avg_studies_per_patient,min_studies_per_patient,max_studies_per_patient
0,91372,2.259861,1,53


In [81]:
common_diag_echo_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(echo_study_result["subject_id"])]
common_diag_echo_patients

,subject_id
0,10923578
1,10979480
2,11023120
3,11030142
4,11036723
...,...
48423,15438712
48427,19394690
48433,19561115
48434,19675353


# Look for dangerous LVEF counts in echocardiography dataset

In [101]:
LVEF_study_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("exploration_sql_files/echo_sql/LVEF_counts.sql", "r") as f:
    sql: str = f.read()

LVEF_study_result: pd.DataFrame = LVEF_study_con.execute(sql).df() # remember to fetch result using df()
LVEF_study_result

,n_patients_dangerous_lvef,avg_dangerous_lvef_studies_per_patient,min_dangerous_lvef_studies_per_patient,max_dangerous_lvef_studies_per_patient
0,15158,1.92156,1,28


In [95]:
common_diag_LVEF_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(LVEF_study_result["subject_id"])]
common_diag_LVEF_patients

,subject_id
3,11030142
7,10958187
20,11146299
28,11124859
36,11234187
...,...
48346,18080257
48350,18191909
48379,11742505
48380,11783844


# Count number of ECG studies

In [ ]:
ECG_study_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open(f"exploration_sql_files/ecg_sql/ECG_counts.sql", "r") as f:
    sql: str = f.read()

ECG_study_result: pd.DataFrame = ECG_study_con.execute(sql).df() # remember to fetch result using df()
ECG_study_result

,num_patients,mean_ecgs,median_ecgs,min_ecgs,max_ecgs,std_ecgs
0,161352,4.958321,2.0,1,260,8.075239


In [149]:
common_diag_ECG_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(ECG_study_result["subject_id"])]
common_diag_ECG_patients

,subject_id
0,10923578
1,10979480
2,11023120
3,11030142
4,11036723
...,...
48432,19542419
48433,19561115
48434,19675353
48436,19624253
